# Central Limit Theorem with Transaction Amounts

**Dataset:** `banking_operations.csv`  
**Tools:** pandas, NumPy, Matplotlib and topic-specific statistical/ML functions  

This notebook explains the concept in simple terms and connects every calculation to banking operations.

## 1. Central Limit Theorem in simple terms

If we repeatedly take random samples of the same size and calculate their means, those means tend to form an approximately normal distribution as sample size increases—even when individual transaction amounts are not normal.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

df = pd.read_csv("banking_operations.csv")
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

print("Dataset shape:", df.shape)
display(df.head())

## 2. Examine the original transaction amounts

The Central Limit Theorem does not require the original observations themselves to become normally distributed.

In [ ]:
population_mean = df["Amount"].mean()
population_std = df["Amount"].std(ddof=0)
print(f"Population mean in this teaching dataset: {population_mean:,.2f}")
print(f"Population standard deviation: {population_std:,.2f}")

df["Amount"].plot(kind="hist", bins=10, edgecolor="black", color="#F2994A", figsize=(8, 4))
plt.title("Original Transaction Amount Distribution")
plt.xlabel("Amount")
plt.show()

## 3. Repeated sampling with pandas

Because the teaching CSV is small, sampling is performed with replacement. Each sample mean represents an estimate of the average transaction amount.

In [ ]:
def sample_means(series, sample_size, repetitions=2000, random_seed=42):
    random_generator = np.random.default_rng(random_seed)
    means = []
    for _ in range(repetitions):
        sample_seed = int(random_generator.integers(0, 1_000_000))
        sample = series.sample(n=sample_size, replace=True, random_state=sample_seed)
        means.append(sample.mean())
    return pd.Series(means, name=f"Sample means, n={sample_size}")

means_5 = sample_means(df["Amount"], 5)
means_15 = sample_means(df["Amount"], 15)
means_30 = sample_means(df["Amount"], 30)

display(pd.DataFrame({
    "Sample size": [5, 15, 30],
    "Mean of sample means": [means_5.mean(), means_15.mean(), means_30.mean()],
    "SD of sample means": [means_5.std(ddof=1), means_15.std(ddof=1), means_30.std(ddof=1)]
}))

## 4. Visual comparison

Larger samples produce a narrower distribution of sample means around the population mean.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for axis, values, n in zip(axes, [means_5, means_15, means_30], [5, 15, 30]):
    values.plot(kind="hist", bins=25, edgecolor="black", alpha=0.8, ax=axis)
    axis.axvline(population_mean, color="red", linestyle="--")
    axis.set_title(f"Sample Means: n={n}")
    axis.set_xlabel("Mean Amount")
plt.tight_layout()
plt.show()

## 5. Standard error

The standard error is the expected spread of sample means. It decreases as sample size increases.

In [ ]:
standard_error_table = pd.DataFrame({"Sample_Size": [5, 15, 30]})
standard_error_table["Theoretical_SE"] = (
    population_std / np.sqrt(standard_error_table["Sample_Size"])
)
standard_error_table["Simulated_SE"] = [
    means_5.std(ddof=1), means_15.std(ddof=1), means_30.std(ddof=1)
]
display(standard_error_table)

## 6. Approximate confidence interval for the mean

Using the CLT, an approximate 95% interval is sample mean ± 1.96 × standard error.

In [ ]:
n = len(df)
sample_mean = df["Amount"].mean()
sample_std = df["Amount"].std(ddof=1)
standard_error = sample_std / np.sqrt(n)
lower = sample_mean - 1.96 * standard_error
upper = sample_mean + 1.96 * standard_error

display(pd.Series({
    "Sample Mean": sample_mean,
    "Standard Error": standard_error,
    "Approximate 95% Lower Limit": lower,
    "Approximate 95% Upper Limit": upper
}).to_frame("Value"))

## Banking interpretation and cautions

The CLT supports estimation of average transaction amounts, processing times or daily volumes. It applies to sample means—not necessarily individual transactions. Randomness, independence and representative sampling still matter; larger samples do not repair biased data.